<a href="https://colab.research.google.com/github/MParvan/ecg-biometrics-bench/blob/main/experiments/run_Module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluation protocols

The framework implements eight protocols, formed by crossing three
binary choices:

| | Same session | Cross session |
|---|---|---|
| **Seen subjects** | tasks 1, 2 | tasks 5, 6 |
| **Unseen subjects** | tasks 3, 4 | tasks 7, 8 |

Odd numbers are identification (1:N), even are verification (1:1).

Difficulty increases left to right and top to bottom. Task 1 is the
protocol most of the literature reports; task 8 is closest to
deployment.

In [ ]:
# Clone the framework and install its dependencies.
# On Colab this takes two to three minutes, mostly PyTorch.
!git clone https://github.com/MParvan/ecg-biometrics-bench.git
%cd ecg-biometrics-bench
!pip install -q -r requirements.txt

## Running a protocol from the command line

This is the normal entry point. Every option is recorded in the
experiment log, so a result can always be traced to what produced it.

In [ ]:
!python main.py \
  --dataset ecgid \
  --task 1 \
  --data_split_mode all-available \
  --epochs 10 --n_runs 1 --save_results

## The same protocol, subject-disjoint

Task 3 trains on one set of identities and evaluates on a completely
different set. It measures whether the representation generalises,
rather than whether the model memorised its training subjects.

Tasks 3, 4, 7, and 8 require `--use_template`, because there is no
classifier head for identities the model never saw.

In [ ]:
!python main.py \
  --dataset ecgid \
  --task 3 \
  --data_split_mode all-available \
  --use_template \
  --epochs 10 --n_runs 1 --save_results

## Cross-session

Task 5 enrols and probes from different recordings, introducing the
temporal drift that same-session evaluation hides.

In [ ]:
!python main.py \
  --dataset ecgid \
  --task 5 \
  --data_split_mode leave-last-out-long-term \
  --use_template \
  --template_fusion_method mean \
  --epochs 10 --n_runs 1 --save_results

## Calling the runners directly

Useful when you want the returned metrics in the notebook rather
than a log file. Every runner returns its metrics as a tuple.

In [ ]:
import numpy as np
from load_dataset import load_ecgid_dataset
from models import DeepECG
from run import run_closed_set_verification

x, y = load_ecgid_dataset(data_split_mode='all-available').load_all_data()

eer, auc, d_prime, tar = run_closed_set_verification(
    x, y, DeepECG,
    epochs=10,
    use_template=True,
    num_pairs=5000,
    sampling_mode='balanced',
    save_results_and_settings=False,
)

print(f'EER        {eer:.4f}')
print(f'AUC        {auc:.4f}')
print(f"d-prime    {d_prime:.4f}")
print(f'TAR@0.1%FAR {tar:.4f}')

## Reporting more than one operating point

Verification tasks report TAR at several false-acceptance rates by
default, and the full ROC, DET, and CMC curves are written alongside
the metrics. `--target_fars` changes which points are reported.

In [ ]:
!python main.py \
  --dataset ecgid \
  --task 2 \
  --data_split_mode all-available \
  --use_template \
  --target_fars 0.01 0.001 0.0001 \
  --epochs 10 --n_runs 1 --save_results

## Reproducing published results

The notebooks above use reduced settings. The reported tables come
from the configurations in `configs/paper_reproduction/`, which are
the source of truth for what was run:

In [ ]:
!python -m scripts.reproduce_tables --table 5 --dry-run